# Analysing an EVM account

Sync one Sepolia account into a temporary database, then look at its activity with pandas. History needs `BLOCKSCOUT_API_KEY` in the environment or in `.env`.

In [ ]:
import tempfile
from pathlib import Path

from multichain_indexer import Indexer

ADDRESS = "0xAA37651031A4b641962550415dB2C43E8d962232"
folder = tempfile.TemporaryDirectory()
indexer = Indexer("ethereum", database=Path(folder.name) / "example.db")
result = indexer.sync(ADDRESS)
result

Exports carry the shared fields as columns and the `native` block as JSON. `amount` and `fee` are text in wei, because a wei amount does not fit in 64 bits.

In [ ]:
import io
import json

import pandas as pd

frame = pd.read_csv(io.StringIO(indexer.export("csv", account=result.account)), dtype={"fee": str})
native = frame["native"].map(json.loads)
frame["kind"] = native.map(lambda n: n["kind"])
frame["gas_used"] = native.map(lambda n: n["gas_used"])
frame["fee_gwei"] = frame["fee"].map(lambda fee: int(fee) / 10**9)
frame[["time", "kind", "status", "gas_used", "fee_gwei"]].head()

In [ ]:
frame.groupby("kind")[["gas_used", "fee_gwei"]].agg(["count", "sum", "max"])

In [ ]:
indexer.metrics(days=3650, account=result.account).native

In [ ]:
indexer.close()
folder.cleanup()